# Migration Phase 8: full pipeline walkthrough

Phase 8 is the final phase of the data pipeline migration
(`data_pipeline_migration_plan.md`). It updates the CLI entry point to the
new configuration schema and removes notebooks/scripts left over from the
pre-migration loaders:

- **`neuroalign.data.preprocessing.cli`** rewritten for the new
  `PipelineConfig`/`DataPaths`/`ModalityConfig` (brainlink + tabular
  derivatives, `--brainlink-db`, `--tabular-derivatives-root`,
  `--atlas-name`, `--anat-atlases`, `--session-variant`, `--labs`, ...).
- Removed obsolete `test_import.py` (exercised the deleted
  `AnatomicalLoader`/`DiffusionLoader`/`QuestionnaireLoader`).
- Removed pre-migration notebooks `01_test_data_loading.ipynb` and
  `02_explore_feature_store.ipynb` (CAT12/QSIPrep-based, fully superseded by
  `migration_00`-`migration_07`).
- Removed unused `scripts/cat12_tiv_template.m` (MATLAB TIV calculation,
  removed in Phase 6).

This notebook is the end-to-end walkthrough: run the CLI to build a fresh
`FeatureStore`, then feed it into both the univariate and multivariate
regional BAG estimators.

## 1. CLI help

`neuroalign-prepare` (= `python -m neuroalign.data.preprocessing.cli`) now
exposes the brainlink + tabular-derivatives configuration.

In [1]:
import subprocess
from pathlib import Path

project_root = Path.cwd().parent
help_text = subprocess.run(
    ["python", "-m", "neuroalign.data.preprocessing.cli", "--help"],
    capture_output=True, text=True, cwd=project_root,
).stdout
print(help_text)

usage: cli.py [-h] [--brainlink-db BRAINLINK_DB]
              [--tabular-derivatives-root TABULAR_DERIVATIVES_ROOT]
              [--output OUTPUT] [--no-anatomical] [--no-diffusion]
              [--prefix PREFIX] [--compression {snappy,gzip,brotli,none}]
              [--atlas-name ATLAS_NAME] [--anat-atlases ANAT_ATLASES]
              [--session-variant {cross,plain,subject}]
              [--labs LABS [LABS ...]] [--allow-incomplete-mapping]
              [--verbose] [--log-file LOG_FILE] [--force]

Prepare NeuroAlign feature matrices from neuroimaging data

options:
  -h, --help            show this help message and exit
  --atlas-name ATLAS_NAME
                        Atlas name (env: ATLAS_NAME, default:
                        Schaefer2018N400n7Tian2020S2)
  --anat-atlases ANAT_ATLASES
                        Comma-separated anatomical atlas folders, cortex first
                        (env: ANAT_ATLASES, default:
                        Schaefer2018N400n7,Tian2020S2)
  --s

## 2. Run the pipeline end-to-end via the CLI

Build a fresh `FeatureStore` for lab `TS` under `data/processed/migration_demo`
(gitignored).

In [2]:
import shutil
from pathlib import Path

project_root = Path.cwd().parent
demo_dir = project_root / "data" / "processed" / "migration_demo"
shutil.rmtree(demo_dir, ignore_errors=True)

run = subprocess.run(
    [
        "python", "-m", "neuroalign.data.preprocessing.cli",
        "--output", str(demo_dir),
        "--labs", "TS",
    ],
    capture_output=True, text=True, cwd=project_root,
)
print(run.stdout[-2000:])
assert run.returncode == 0, run.stderr

rix3actHSVS_mtnorm_inliermask_z_filtered_mean
    - MRtrix3actHSVS_mtnorm_inliermask_z_filtered_std
    - MRtrix3actHSVS_mtnorm_inliermask_iqr_filtered_mean
    - MRtrix3actHSVS_mtnorm_inliermask_iqr_filtered_std
    - MRtrix3actHSVS_mtnorm_inliermask_skewness
    - MRtrix3actHSVS_mtnorm_inliermask_excess_kurtosis
    - MRtrix3actHSVS_mtnorm_inliermask_percentile_5
    - MRtrix3actHSVS_mtnorm_inliermask_percentile_25
    - MRtrix3actHSVS_mtnorm_inliermask_percentile_75
    - MRtrix3actHSVS_mtnorm_inliermask_percentile_95
    - MRtrix3actHSVS_mtnorm_inliermask_coverage
    - MRtrix3actHSVS_mtnorm_inliermask_volume_mm3
    - MRtrix3actHSVS_mtnorm_inliermask_voxel_count
    - MRtrix3actHSVS_mtnorm_norm_mean
    - MRtrix3actHSVS_mtnorm_norm_std
    - MRtrix3actHSVS_mtnorm_norm_median
    - MRtrix3actHSVS_mtnorm_norm_sum
    - MRtrix3actHSVS_mtnorm_norm_cv
    - MRtrix3actHSVS_mtnorm_norm_robust_mean
    - MRtrix3actHSVS_mtnorm_norm_robust_std
    - MRtrix3actHSVS_mtnorm_norm_robust_cv
    

## 3. Load the resulting `FeatureStore`

In [3]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")

from neuroalign.data.preprocessing import FeatureStore

store = FeatureStore(demo_dir)
meta = store.load_metadata()
print(f"n_sessions={len(meta)}, n_subjects={meta['uid'].nunique()}")
print(f"n_wide_features={len(store.list_features())}")
meta.head()

n_sessions=46, n_subjects=42
n_wide_features=1278


,session_id,uid,subject_code,subject_code_bids,lab,scan_date,scan_tag,scan_number,protocol,study,group_label,mapping_complete,AGE,sex,weight_kg,height_m,dominant_hand
0,202410131245,S076379,BB01028,None,TS,2024-10-13,None,2,Brain Bank,TS,None,True,22.26,Female,67.0,1.61,None
1,202410071918,S606615,BB01023,None,TS,2024-10-07,None,1,Brain Bank,TS,None,True,30.74,Male,85.0,1.74,None
2,202410101257,S656985,BB01025,None,TS,2024-10-10,None,1,Brain Bank,TS,None,True,28.80,Female,59.0,1.58,None
3,202510160926,S108449,BB01018,None,TS,2025-10-16,None,2,Brain Bank,TS,None,True,31.28,Female,80.0,1.58,None
4,202507241349,S028694,BB01012,None,TS,2025-07-24,None,3,Brain Bank,TS,None,True,23.81,Female,67.0,1.64,None


## 4. Univariate regional BAG

One Ridge model per region, predicting age from that region's cortical
thickness + sex + TIV.

In [4]:
from neuroalign.modeling import BAGConfig, RegionalBAGEstimator

features = store.load_feature("anat_thickness_mean_mm", include_metadata=False)
metadata = store.load_metadata().merge(store.load_tiv(), on=["uid", "session_id"], how="left")

config = BAGConfig(
    age_col="AGE",
    sex_col="sex",
    tiv_col="tiv_mm3",
    subject_col="uid",
    session_col="session_id",
)

univariate = RegionalBAGEstimator(config)
uni_result = univariate.fit_predict(features, metadata)
uni_result.region_metrics.sort_values("r2", ascending=False).head(10)

Fold 1/5:   0%|          | 0/400 [00:00<?, ?it/s]

Fold 2/5:   0%|          | 0/400 [00:00<?, ?it/s]

Fold 3/5:   0%|          | 0/400 [00:00<?, ?it/s]

Fold 4/5:   0%|          | 0/400 [00:00<?, ?it/s]

Fold 5/5:   0%|          | 0/400 [00:00<?, ?it/s]

,region,r2,mae,correlation
9,7Networks_LH_Cont_PFCl_7,0.546185,2.119420,0.745441
203,7Networks_RH_Cont_PFCl_10,0.505161,3.105415,0.748530
330,7Networks_RH_SomMot_1,0.482980,2.695804,0.725260
272,7Networks_RH_DorsAttn_Post_1,0.482015,3.238455,0.733314
350,7Networks_RH_SomMot_28,0.354167,3.226285,0.635345
73,7Networks_LH_Default_pCunPCC_9,0.327298,3.130492,0.658581
237,7Networks_RH_Default_PFCdPFCm_4,0.323154,2.857736,0.645498
21,7Networks_LH_Cont_pCun_2,0.317069,3.509091,0.578891
135,7Networks_LH_SomMot_12,0.314416,3.209243,0.659377
281,7Networks_RH_DorsAttn_Post_18,0.259556,3.589882,0.632709


## 5. Multivariate regional BAG

Combine cortical thickness and diffusion FA into per-region feature blocks
via `regional-stacker`.

In [5]:
from neuroalign.modeling import MultivariateRegionalBAGEstimator

multi_config = BAGConfig(age_col="AGE")
multivariate = MultivariateRegionalBAGEstimator(multi_config)

multi_result = multivariate.fit_predict(
    store, ["anat_thickness_mean_mm", "DSIStudio_tensor_fa_mean"]
)
multi_result.bag.describe()

/home/galkepler/Projects/neuroalign/src/neuroalign/modeling/multivariate/estimator.py:165: UserWarning: 10 subject(s) absent from one or more tables and excluded from the intersection: [('S019804', '202410071148'), ('S166352', '202410080846'), ('S203749', '202411031340'), ('S483889', '202409301535'), ('S593447', '202410310932'), ('S606615', '202410071918'), ('S656985', '202410101257'), ('S736879', '202410301546'), ('S787835', '202411070913'), ('S829190', '202410141135')]
  x, region_mapping, sessions = wide_to_stacker_input(tables)


Outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

,bag
count,1.200000e+01
mean,-7.993606e-15
std,1.311100e+00
min,-1.459974e+00
25%,-1.108034e+00
50%,-1.629357e-01
75%,6.265717e-01
max,2.349811e+00


## 6. Migration complete

All 9 phases of `data_pipeline_migration_plan.md` are now implemented:

0. Dependencies + `.env`
1. `BehavioralLoader` (brainlink)
2-3. `TabularDerivativesLoader` (anatomical + diffusion)
4. `FeatureStore` rewrite (long/wide formats, `META_COLS`)
5. `DataPreparationPipeline` + `PipelineConfig` rewrite
6. Removed obsolete loaders/transformers/deps/`.env` keys
7. `MultivariateRegionalBAGEstimator` via `regional-stacker`
8. CLI + notebook cleanup (this notebook)